# 🎛️ True-LoRA Playground — 好きなプロンプトでLoRAを作るデモ

テキストを打ち込むだけで、その場で **LoRAアダプタ** を生成します。微調整（fine-tuning）は不要。
プロンプト → LoRA を1回のフォワードで生成します。

- **Step 1–3**: セットアップ（ハイパーネットを数秒だけ学習）— 一度だけ実行
- **Step 4**: 🚀 好きなプロンプトでLoRAを作る（何度でも）
- **Step 5–6**: （オプション）生成したLoRAを matutake にマージして実際に動かす

> 生成だけなら **CPUでも数ミリ秒** で動きます。GPUはオプションのマージ＋推論でのみ必要です。

## Step 1: セットアップ

In [ ]:
!pip install git+https://github.com/MARVserver/TrueLora.git -q
# sentence-transformers を入れると多言語の意味エンコーダが有効になり、
# 日本語を含む任意のプロンプトでも近いアダプタを検索できます（無ければ自動でhashingにフォールバック）。
!pip install "sentence-transformers" transformers accelerate bitsandbytes -q
print("Done!")

In [ ]:
import gc
import re
from pathlib import Path

import torch
from true_lora.adapter import AdapterSpec, LoraTensorSpec, save_peft_adapter
from true_lora.generator import TrueLoraGenerator
from true_lora.text import SemanticTextEncoder
from true_lora.train import train_on_adapter_bank
from true_lora.reporting import write_json_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Device: {DEVICE}")

## Step 2: 学習データを用意（Adapter Bank なし）

少数の **(説明文 → LoRA) のペア** を学習データとして用意するだけ。検索データベース（Adapter Bank）は作りません。
ハイパーネットがこの対応を学び、推論時は **プロンプト → LoRA を直接生成** します（検索・ブレンドなしの純 Text-to-LoRA）。

In [ ]:
# LoRAの適用先（matutakeのアーキテクチャに合わせる → 後でそのままマージ可能）
LORA_RANK = 4
LORA_ALPHA = 8.0
MAX_NORM = 4.0

from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("summerMC/matutake", trust_remote_code=True)
HIDDEN_SIZE = cfg.hidden_size
NUM_ATTENTION_HEADS = cfg.num_attention_heads
NUM_KEY_VALUE_HEADS = getattr(cfg, "num_key_value_heads", NUM_ATTENTION_HEADS)
HEAD_DIM = getattr(cfg, "head_dim", HIDDEN_SIZE // NUM_ATTENTION_HEADS)

# GQAモデルでは q_proj と v_proj の出力次元が異なる（KVヘッド数に依存）
PROJ_OUT_FEATURES = {
    "q_proj": NUM_ATTENTION_HEADS * HEAD_DIM,
    "v_proj": NUM_KEY_VALUE_HEADS * HEAD_DIM,
}
LORA_TARGETS = ["q_proj", "v_proj"]
TARGET_LAYERS = [0, 6, 12, 18]

lora_specs = []
for li in TARGET_LAYERS:
    for t in LORA_TARGETS:
        name = f"model.layers.{li}.self_attn.{t}"
        lora_specs.append(LoraTensorSpec(name, PROJ_OUT_FEATURES[t], HIDDEN_SIZE, LORA_RANK, LORA_ALPHA))

print(f"Hidden: {HIDDEN_SIZE} | specs: {len(lora_specs)} | out_features: {PROJ_OUT_FEATURES}")

In [ ]:
# 多言語の意味エンコーダ（オフライン時はhashingに自動フォールバック）
encoder = SemanticTextEncoder()
print(f"Encoder backend: {encoder.backend} | dim: {encoder.dim}")

# 学習データ（説明文, lora_A値, lora_B値, スコア）
adapters_config = [
    ("python code generation", 0.15, 0.08, 0.85),
    ("python function implementation", 0.18, 0.09, 0.82),
    ("algorithm implementation", 0.20, 0.10, 0.88),
    ("data structure implementation", 0.17, 0.08, 0.83),
    ("dynamic programming solutions", 0.22, 0.11, 0.86),
    ("code debugging and error fixing", 0.13, 0.06, 0.79),
    ("unit test writing", 0.15, 0.07, 0.77),
    ("performance optimization", 0.18, 0.09, 0.80),
    ("numpy array operations", 0.16, 0.08, 0.82),
    ("torch deep learning code", 0.19, 0.09, 0.85),
    ("creative writing and storytelling", 0.14, 0.07, 0.75),
    ("japanese translation", 0.12, 0.06, 0.73),
    ("polite business email writing", 0.11, 0.05, 0.72),
    ("math reasoning and proofs", 0.20, 0.10, 0.84),
    ("sql query generation", 0.16, 0.08, 0.80),
]

adapters = []
for desc, sa, sb, score in adapters_config:
    tensors = {}
    for spec in lora_specs:
        tensors[f"{spec.name}.lora_A.weight"] = torch.full(spec.a_shape, sa)
        tensors[f"{spec.name}.lora_B.weight"] = torch.full(spec.b_shape, sb)
    adapters.append(AdapterSpec(
        description=desc,
        embedding=encoder.encode(desc),
        tensors=tensors,
        metrics={"score": score},
    ))

# Adapter Bank は作らない。これはあくまで (説明文 → LoRA) の学習データ。
print(f"学習データ: {len(adapters)} 件（Adapter Bank なし）")

## Step 3: ハイパーネットを学習（一度だけ・数秒）

`adapter_bank=None` の **純 Text-to-LoRA** モード。`hyper_kind="conditioned"` は (タスク, 層, モジュール) 条件付き
ハイパーネットで、検索に頼らずプロンプトから LoRA を生成します。学習は `(説明文 → LoRA)` のペアから直接行います。

In [ ]:
generator = TrueLoraGenerator(
    tensor_specs=lora_specs,
    adapter_bank=None,          # Adapter Bank なし（検索DBを使わない純Text-to-LoRA）
    hidden_dim=256,
    max_tensor_norm=MAX_NORM,
    encoder=encoder,            # 意味エンコーダで生成を駆動
    hyper_kind="conditioned",   # 共有trunk + (タスク,層,モジュール)条件付け
)

# 学習は (説明文 → LoRA) のペアから直接（バンク不要）
losses = train_on_adapter_bank(generator, adapters, steps=200, lr=1e-3)
print(f"Trained: loss {losses[0]:.4f} -> {losses[-1]:.6f}")
print(f"Hypernetwork params: {sum(p.numel() for p in generator.hyper.parameters()):,}")

## 🚀 Step 4: 好きなプロンプトでLoRAを作る

`make_lora(prompt)` にテキストを渡すだけ。確信度（confidence）と保存先を表示し、
`output/lora_*.pt`（HuggingFace PEFT互換）として保存します。Adapter Bank を使わず、
ハイパーネットがプロンプトから直接 LoRA を生成します。

In [ ]:
import hashlib

def _slug(text):
    # Keep readable characters (Unicode word chars incl. 日本語), then append a short
    # hash so distinct prompts never collide on the same filename.
    base = re.sub(r"[^\w]+", "_", text.strip(), flags=re.UNICODE).strip("_")
    digest = hashlib.blake2b(text.encode("utf-8"), digest_size=3).hexdigest()
    return f"{base[:40] or 'lora'}_{digest}"

def make_lora(prompt, k=8, save=True, show=True):
    """任意のプロンプトからLoRAを生成して返す（Adapter Bank 不使用）。"""
    state_dict, report = generator.generate(prompt, retrieval_k=k)
    confidence = 1.0 - report["uncertainty"]
    if show:
        print(f"Prompt: {prompt!r}")
        print(f"  Confidence : {confidence:.1%}  (uncertainty {report['uncertainty']:.3f})")
        # Adapter Bank を使う場合のみ検索結果が入る
        if report["retrieved_adapters"]:
            print("  近いアダプタ:")
            for p in report["retrieved_adapters"][:5]:
                print(f"    [{p['rank']}] {p['description']:<34} w={p['weight']:.3f}")
    path = None
    if save:
        path = OUTPUT_DIR / f"lora_{_slug(prompt)}.pt"
        cpu_state = {kk: vv.cpu() for kk, vv in state_dict.items()}
        save_peft_adapter(path, cpu_state, report)
        write_json_report(OUTPUT_DIR / f"report_{_slug(prompt)}.json", report)
        if show:
            print(f"  Saved      : {path}")
    return state_dict, report, path

print("make_lora(prompt) 準備完了")

In [ ]:
#@title ✍️ ここにプロンプトを入力して実行 { display-mode: "form" }
YOUR_PROMPT = "write a fast vectorized numpy function"  #@param {type:"string"}

state_dict, report, path = make_lora(YOUR_PROMPT)

### いろいろなプロンプトを試す（英語・日本語）

In [ ]:
for p in [
    "python code generation",
    "二分探索のアルゴリズムを書く",
    "creative storytelling in english",
    "丁寧なビジネスメールを書く",
    "optimize a slow training loop",
]:
    make_lora(p, save=False)
    print("-" * 64)

## （オプション）Step 5: 生成したLoRAを matutake にマージして動かす

ここから先は実モデル（2B）を読み込みます。**GPU推奨**。
直前の `make_lora` が保存した `path` を、または `output/` 内の任意の `*.pt` を使えます。

In [ ]:
# メモリ解放（生成器はもう不要）
ADAPTER_PATH = path  # 直前の make_lora の出力。任意の output/*.pt を指定可
assert ADAPTER_PATH is not None, "先に Step 4 を実行してLoRAを生成してください"

del generator, adapters
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

from true_lora.merge import merge_adapter_into_hf_model

merged_dir = OUTPUT_DIR / "matutake_merged"
print(f"Merging {ADAPTER_PATH.name} into matutake ...")
merge_report = merge_adapter_into_hf_model(
    adapter_path=ADAPTER_PATH,
    model_name_or_path="summerMC/matutake",
    output_dir=merged_dir,
    dtype="bfloat16",
    device="cpu",
    allow_download=True,
)
print(f"Done! Output: {merge_report['output_dir']}")
print(f"Applied layers: {merge_report['applied_layers']}")

## Step 6: マージしたモデルで生成

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

load_kwargs = dict(trust_remote_code=True, device_map="auto")
if DEVICE == "cuda":
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
else:
    load_kwargs["torch_dtype"] = torch.float32  # CPUは低速です

print("Loading merged model ...")
tokenizer = AutoTokenizer.from_pretrained(str(merged_dir), trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(str(merged_dir), **load_kwargs)
print("Loaded!")

In [ ]:
def chat(prompt, max_new_tokens=256):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

# Step 4 で入力したプロンプトでそのまま試す
print(chat(YOUR_PROMPT))

## 仕組み（まとめ）

```
プロンプト
   │  SemanticTextEncoder（多言語・意味埋め込み）
   ▼
   Conditioned Hypernetwork（プロンプト → LoRA を直接生成）
   │  ノルムclip
   ▼
   LoRA アダプタ（PEFT互換 .pt）
```

Adapter Bank（検索DB）や検索・ブレンドは使いません。ハイパーネットだけの純 Text-to-LoRA です。

| 項目 | 値 |
| --- | --- |
| Adapter Bank | なし（`adapter_bank=None`） |
| Encoder | SemanticTextEncoder（多言語, hashingフォールバック） |
| Hypernetwork | conditioned (task, layer, module) |
| LoRA Rank / Alpha | 4 / 8.0 |
| Target | q_proj, v_proj × layers {0,6,12,18} |
| 生成 | プロンプト → LoRA を1フォワード（CPUで数ms） |

`make_lora("好きなプロンプト")` を何度でも呼べます。